# 25 · Query Patterns & Recipes

A toolkit of real-world patterns that combine what you've learned. These come up
constantly in analytics and application code:
- **Top-N per group**
- **Deduplication** (keep one row per key)
- **Gaps & islands** (find consecutive runs)
- **Pivot / unpivot**
- **Running % of total** (Pareto/cumulative share)
- **Date spine** to fill missing dates

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Top-N per group
"Top 2 most expensive products in each category." Rank within each partition,
then keep the top ranks. This is the go-to pattern.

In [ ]:
%%sql
WITH ranked AS (
    SELECT category_id, product_name, unit_price,
           ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY unit_price DESC) AS rn
    FROM products
)
SELECT category_id, product_name, unit_price
FROM ranked
WHERE rn <= 2
ORDER BY category_id, unit_price DESC;

## Deduplication — keep one row per key
Given duplicates, keep exactly one (e.g. the earliest signup per country). Same
`ROW_NUMBER` idea, keep `rn = 1`.

In [ ]:
%%sql
WITH ranked AS (
    SELECT country, first_name, signup_date,
           ROW_NUMBER() OVER (PARTITION BY country ORDER BY signup_date) AS rn
    FROM customers
)
SELECT country, first_name AS first_signup, signup_date
FROM ranked
WHERE rn = 1
ORDER BY country;

## Gaps & islands
Find consecutive runs in a sequence. The trick: `value - ROW_NUMBER()` is
constant within a consecutive run, so group by that difference.

In [ ]:
%%sql
WITH nums(n) AS (VALUES (1), (2), (3), (5), (6), (9), (10), (11)),
     grouped AS (
        SELECT n, n - ROW_NUMBER() OVER (ORDER BY n) AS grp
        FROM nums
     )
SELECT MIN(n) AS run_start, MAX(n) AS run_end, COUNT(*) AS length
FROM grouped
GROUP BY grp
ORDER BY run_start;

## Pivot (rows → columns)
Order counts by status, one column each (conditional aggregation):

In [ ]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE status = 'completed') AS completed,
    COUNT(*) FILTER (WHERE status = 'pending')   AS pending,
    COUNT(*) FILTER (WHERE status = 'cancelled') AS cancelled
FROM orders;

## Unpivot (columns → rows)
Turn a product's two metrics into a tall (name, metric, value) shape with
`UNION ALL`:

In [ ]:
%%sql
SELECT product_name, 'price' AS metric, unit_price AS value FROM products
UNION ALL
SELECT product_name, 'stock', in_stock FROM products
ORDER BY product_name, metric
LIMIT 8;

## Running % of total (Pareto)
Cumulative share of revenue by category — which categories make up the bulk?

In [ ]:
%%sql
WITH cat AS (
    SELECT c.category_name, SUM(oi.quantity * oi.unit_price) AS revenue
    FROM order_items oi
    JOIN products p   ON oi.product_id = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY c.category_name
)
SELECT category_name,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY revenue DESC)
             / SUM(revenue) OVER (), 1) AS cumulative_pct
FROM cat
ORDER BY revenue DESC;

## Date spine — fill missing dates
Reports need a row for *every* day, even days with no orders. Generate a date
series with a recursive CTE, then `LEFT JOIN` the data onto it.

In [ ]:
%%sql
WITH RECURSIVE days(day) AS (
    SELECT '2024-02-01'
    UNION ALL
    SELECT DATE(day, '+1 day') FROM days WHERE day < '2024-02-05'
)
SELECT days.day,
       COUNT(o.order_id) AS orders
FROM days
LEFT JOIN orders o ON o.order_date = days.day
GROUP BY days.day
ORDER BY days.day;

## Practice

**✏️ Exercise 1.** Find the single most recent order per customer (order_id, customer_id, order_date) using the top-N-per-group pattern with N=1.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH ranked AS (
  SELECT order_id, customer_id, order_date,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) AS rn
  FROM orders
)
SELECT order_id, customer_id, order_date FROM ranked WHERE rn = 1
ORDER BY customer_id;

**✏️ Exercise 2.** Produce a running percentage-of-total of revenue by customer (top spenders first).

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH rev AS (
  SELECT o.customer_id, SUM(oi.quantity * oi.unit_price) AS revenue
  FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
  GROUP BY o.customer_id
)
SELECT customer_id, ROUND(revenue,2) AS revenue,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY revenue DESC) / SUM(revenue) OVER (), 1) AS cume_pct
FROM rev ORDER BY revenue DESC;

### ✅ Recap
These patterns — top-N per group, dedup, gaps & islands, pivot/unpivot, running
%, and date spines — solve a huge share of real analytical questions by combining
window functions, CTEs, and conditional aggregation.

**Next:** `26_advanced_capstone.ipynb`.